In [1]:
# === SESSION BOOTSTRAP (run first in every notebook) ======================
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys

PARENT = "/content/drive/MyDrive/UAV_TRUST_Research"
REPO   = f"{PARENT}/uav-trust-research"

for fname in (".gitconfig", ".git-credentials"):
    src = os.path.join(PARENT, fname)
    if os.path.exists(src):
        subprocess.run(f'cp "{src}" /root/{fname}', shell=True)
subprocess.run("git config --global credential.helper store", shell=True)

r = subprocess.run("git config --global user.name && git config --global user.email",
                   shell=True, capture_output=True, text=True)
print("git identity:", r.stdout.strip() or "MISSING - run 00_setup.ipynb first")

if os.path.isdir(REPO):
    os.chdir(REPO)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    print("cwd:", os.getcwd())
else:
    print("Repository not on Drive yet - run 00_setup.ipynb first.")

Mounted at /content/drive
git identity: Md Anas Biswas
anasbiswas@gmail.com
cwd: /content/drive/MyDrive/UAV_TRUST_Research/uav-trust-research


In [2]:
!pip install xgboost shap scikit-learn matplotlib pandas numpy scipy requests --quiet

In [3]:
# Configuration: transfer-fragility across seeds (robustness of the mechanism from notebook 05)
CONFIG = {
    "zenodo_record": "15336998",
    "data_dir": "data/uavids2025",
    "label_col": "label",
    "normal_value": "Normal Traffic",
    "drop_col_patterns": ["unnamed", "flowid", "srcaddr", "dstaddr",
                           "srcport", "dstport", "index", "timestamp"],
    "seeds": list(range(10)),
    "n_shap": 2000,
    "normal_fracs": {"train": 0.60, "cal": 0.20, "test_seen": 0.10, "test_shift": 0.10},
    "family_fracs": {"train": 0.60, "cal": 0.20, "test_seen": 0.20},
    "xgb": {"n_estimators": 300, "max_depth": 6, "learning_rate": 0.1,
            "subsample": 0.9, "colsample_bytree": 0.9, "tree_method": "hist"},
    "fig_dir": "figures",
    "report_dir": "reports",
}
for d in [CONFIG["data_dir"], CONFIG["fig_dir"], CONFIG["report_dir"]]:
    os.makedirs(d, exist_ok=True)
CONFIG

{'zenodo_record': '15336998',
 'data_dir': 'data/uavids2025',
 'label_col': 'label',
 'normal_value': 'Normal Traffic',
 'drop_col_patterns': ['unnamed',
  'flowid',
  'srcaddr',
  'dstaddr',
  'srcport',
  'dstport',
  'index',
  'timestamp'],
 'seeds': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
 'n_shap': 2000,
 'normal_fracs': {'train': 0.6,
  'cal': 0.2,
  'test_seen': 0.1,
  'test_shift': 0.1},
 'family_fracs': {'train': 0.6, 'cal': 0.2, 'test_seen': 0.2},
 'xgb': {'n_estimators': 300,
  'max_depth': 6,
  'learning_rate': 0.1,
  'subsample': 0.9,
  'colsample_bytree': 0.9,
  'tree_method': 'hist'},
 'fig_dir': 'figures',
 'report_dir': 'reports'}

In [4]:
# Imports
import numpy as np, pandas as pd, requests, glob
import matplotlib.pyplot as plt
import xgboost as xgb, shap
from scipy.stats import spearmanr
from sklearn.metrics import balanced_accuracy_score
from src.data import load_csvs, detect_schema, prepare_splits
print("imports ready")

imports ready


In [5]:
# Ensure dataset present, then load
rec = CONFIG["zenodo_record"]
if not glob.glob(os.path.join(CONFIG["data_dir"], "**/*.csv"), recursive=True):
    meta = requests.get(f"https://zenodo.org/api/records/{rec}", timeout=60).json()
    for f in meta.get("files", []):
        name, url = f["key"], f["links"]["self"]
        if name.lower().endswith((".csv", ".zip", ".gz")):
            open(os.path.join(CONFIG["data_dir"], name), "wb").write(requests.get(url, timeout=600).content)
    for z in glob.glob(os.path.join(CONFIG["data_dir"], "*.zip")):
        import zipfile; zipfile.ZipFile(z).extractall(CONFIG["data_dir"])
df = load_csvs(CONFIG["data_dir"])
label_col, normal_value, families = detect_schema(df, CONFIG["label_col"], CONFIG["normal_value"])
print("shape:", df.shape, "| families:", families)

shape: (122171, 23) | families: ['Sybil Attack', 'Blackhole Attack', 'Wormhole Attack', 'Flooding Attack']


In [6]:
# Fragility (sign-flip attribution reversal) and shift balanced accuracy for one (family, seed)
def sample(A, n, seed):
    if len(A) > n:
        j = np.random.default_rng(seed).choice(len(A), n, replace=False); return A[j]
    return A

def mean_shap(expl, Xg):
    s = np.asarray(expl.shap_values(Xg))
    if s.ndim == 3:
        s = s[..., 1]
    return s.mean(0)

def fragility_once(df, label_col, normal_value, F, seed, cfg):
    S = prepare_splits(df, label_col, normal_value, F, cfg["drop_col_patterns"],
                       cfg["normal_fracs"], cfg["family_fracs"], seed)
    clf = xgb.XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                            random_state=seed, **cfg["xgb"]).fit(S["X_train"], S["y_train"])
    pr_h = (clf.predict_proba(S["X_shift"])[:, 1] >= 0.5).astype(int)
    balacc = balanced_accuracy_score(S["y_shift"], pr_h)
    Xa = sample(S["X_seen"][np.isin(S["fam_seen"], S["seen_families"])], cfg["n_shap"], seed)
    Xh = sample(S["X_shift"][S["fam_shift"] == F], cfg["n_shap"], seed)
    expl = shap.TreeExplainer(clf)
    m_seen = mean_shap(expl, Xa)
    m_held = mean_shap(expl, Xh)
    flip = np.maximum(m_seen, 0.0) * np.maximum(-m_held, 0.0)
    return {"held_out": F, "seed": seed,
            "shift_balacc": balanced_accuracy_score(S["y_shift"], pr_h),
            "fragility": float(flip.sum())}

In [ ]:
# Run every family across every seed
rows = []
for seed in CONFIG["seeds"]:
    for F in families:
        rows.append(fragility_once(df, label_col, normal_value, F, seed, CONFIG))
    print("seed done:", seed)
raw = pd.DataFrame(rows)
print("runs:", raw.shape[0])

seed done: 0
seed done: 1
seed done: 2
seed done: 3
seed done: 4
seed done: 5
seed done: 6
seed done: 7
seed done: 8


In [ ]:
# Per-family fragility across seeds (mean and std), reported next to balanced accuracy
agg = raw.groupby("held_out").agg(frag_mean=("fragility", "mean"), frag_std=("fragility", "std"),
                                   balacc_mean=("shift_balacc", "mean"), balacc_std=("shift_balacc", "std"))
agg = agg.sort_values("balacc_mean")
out = pd.DataFrame(index=agg.index)
out["fragility"] = agg["frag_mean"].round(3).astype(str) + " ± " + agg["frag_std"].round(3).astype(str)
out["shift_balacc"] = agg["balacc_mean"].round(4).astype(str) + " ± " + agg["balacc_std"].round(4).astype(str)
print(out.to_string())
raw.to_csv(os.path.join(CONFIG["report_dir"], "06_fragility_seed_raw.csv"), index=False)
agg.round(6).to_csv(os.path.join(CONFIG["report_dir"], "06_fragility_seed_aggregate.csv"))

In [ ]:
# Does fragility predict the accuracy ranking within each seed? Report Spearman per seed.
rhos = []
for seed in CONFIG["seeds"]:
    d = raw[raw["seed"] == seed]
    rhos.append(spearmanr(d["fragility"], d["shift_balacc"]).correlation)
rhos = np.array(rhos, float)
print("Spearman(fragility, shift balanced accuracy) per seed:", np.round(rhos, 2).tolist())
print("mean = %.3f  std = %.3f  (strong negative and stable => mechanism holds across draws)"
      % (np.nanmean(rhos), np.nanstd(rhos)))

In [ ]:
# Figure: per-family fragility across seeds (is the ranking stable?)
fams = list(agg.index)
labels = [str(f).replace(" Attack", "") for f in fams]
jit = np.random.default_rng(0)
plt.figure(figsize=(8, 4.5))
for i, F in enumerate(fams):
    v = raw.loc[raw["held_out"] == F, "fragility"].values
    plt.scatter(i + jit.uniform(-0.12, 0.12, len(v)), v, s=22, alpha=0.6, color="#c62828")
    plt.scatter([i], [v.mean()], marker="_", s=800, color="black", zorder=3)
plt.xticks(range(len(fams)), labels)
plt.ylabel("transfer-fragility score")
plt.title("UAVIDS-2025  |  transfer-fragility per family across %d seeds  |  black bar = mean" % len(CONFIG["seeds"]))
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["fig_dir"], "06_fragility_spread.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Commit results (end-of-unit discipline)
!git add reports/ figures/ notebooks/
!git commit -m "06 UAVIDS-2025: transfer-fragility across 10 seeds (mean and std per family, per-seed Spearman)"
!git push origin main